# 🧪 Semana 4 · Unidad 2 — Laboratorio: Union-Find

---

> **Curso:** Estructuras de Datos y Algoritmos  
> **Duración:** 60 minutos  
> **Modalidad:** Individual o en parejas

---

### 📋 Instrucciones generales

- Completa cada celda donde aparezca `# TODO`.
- **No borres** las celdas de prueba automática — te dirán si tu solución es correcta.
- Las pruebas usan `unittest`, la librería estándar de Python (ver S00 · Unit Testing). Cada prueba termina en `ok`, `FAIL` (resultado incorrecto) o `ERROR` (tu código lanzó una excepción); la celda cierra con `OK` cuando todas pasan.
- Puedes (y debes) agregar celdas propias para experimentar.
- Al finalizar, exporta el notebook como **`.ipynb`** y súbelo al aula virtual.

### 🗺️ Estructura del laboratorio

| Parte | Tema | Tiempo |
|-------|------|--------|
| **Setup** | Importaciones y código base | 5 min |
| **Parte 1** | Trazar Quick Find a mano | 10 min |
| **Parte 2** | Completar Quick Find | 15 min |
| **Parte 3** | Completar Quick Union | 15 min |
| **Parte 4** | Análisis comparativo | 10 min |
| **Bonus** | Mejora: Weighted Quick Union | 5+ min |

---

## ⚙️ Setup — Ejecuta esta celda primero

In [ ]:
# Importaciones necesarias para todo el laboratorio
from abc import ABC, abstractmethod
import random
import time
import unittest

# ─── Clase base: NO modificar ───────────────────────────────────────────────
class UnionFind(ABC):
    """TDA Union-Find. Define la interfaz que DEBEN implementar Quick Find y Quick Union."""

    @abstractmethod
    def union(self, p: int, q: int) -> None:
        pass

    @abstractmethod
    def find(self, p: int) -> int:
        pass

    def connected(self, p: int, q: int) -> bool:
        """Implementado aquí usando find() — heredado por todas las subclases."""
        return self.find(p) == self.find(q)

    @abstractmethod
    def count(self) -> int:
        pass

# ─── Visualización en texto ─────────────────────────────────────────────────

def mostrar_id(id_arr, titulo="", anterior=None):
    """
    Imprime id[] alineado bajo sus índices y agrupa los objetos por componente.

    Parámetros:
        id_arr (list[int]): arreglo id[] de Quick Find.
        titulo (str): encabezado opcional.
        anterior (list[int] | None): estado previo; si se entrega, marca con ↑
            las posiciones que cambiaron.
    """
    ancho = len(str(max(len(id_arr) - 1, *id_arr)))

    def fila(valores):
        return "  ".join(f"{v:>{ancho}}" for v in valores)

    componentes = {}
    for i, comp in enumerate(id_arr):
        componentes.setdefault(comp, []).append(i)

    if titulo:
        print(titulo)
    print(f"  índice: {fila(range(len(id_arr)))}")
    print(f"  id[]:   {fila(id_arr)}")
    if anterior is not None:
        print(f"          {fila('↑' if a != b else ' ' for a, b in zip(anterior, id_arr))}")
    print("  " + " ".join("{" + ",".join(map(str, g)) + "}" for g in componentes.values()))

def mostrar_bosque(id_arr, titulo=""):
    """
    Imprime el bosque de Quick Union como el comando `tree` muestra carpetas.

    Cada raíz (id[r] == r) encabeza su árbol entre corchetes y los hijos cuelgan
    con ├── y └──. La sangría de un nodo es su profundidad: los pasos que da
    find() para llegar a la raíz.

    Parámetros:
        id_arr (list[int]): arreglo id[] de Quick Union.
        titulo (str): encabezado opcional.

    Complejidad:
        Temporal: O(n) — cada nodo se imprime una vez.
        Espacial: O(n) — listas de hijos y pila de recursión.
    """
    hijos = {i: [] for i in range(len(id_arr))}
    raices = []
    for i, padre in enumerate(id_arr):
        if padre == i:
            raices.append(i)
        else:
            hijos[padre].append(i)

    def altura(nodo):
        return max((1 + altura(h) for h in hijos[nodo]), default=0)

    def imprimir_hijos(nodo, sangria):
        for k, hijo in enumerate(hijos[nodo]):
            ultimo = k == len(hijos[nodo]) - 1
            print(sangria + ("└── " if ultimo else "├── ") + str(hijo))
            imprimir_hijos(hijo, sangria + ("    " if ultimo else "│   "))

    if titulo:
        print(titulo)
    for r in raices:
        print(f"[{r}]")
        imprimir_hijos(r, "")
    print(f"{len(raices)} árbol(es) · altura máxima {max(altura(r) for r in raices)}")

def mostrar_barras(filas, unidad="ms", ancho=40):
    """
    Imprime un gráfico de barras horizontal: la barra del valor máximo mide `ancho`.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra más larga.
    """
    maximo = max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:8.2f} {unidad}")


print("✅ Setup completo — puedes continuar con la Parte 1")

---

## Parte 1 — Trazar Quick Find a mano  ✏️ *(10 min)*

Dado el estado inicial para **N = 8 objetos**:

```
índice:  0  1  2  3  4  5  6  7
id[]:    0  1  2  3  4  5  6  7
```

Aplica **a mano** las siguientes operaciones de Quick Find y completa la tabla.
Recuerda la regla: `union(p, q)` cambia **todos** los `id[i] == id[p]` al valor `id[q]`.

| Operación | id[0] | id[1] | id[2] | id[3] | id[4] | id[5] | id[6] | id[7] | # comps |
|-----------|-------|-------|-------|-------|-------|-------|-------|-------|---------|
| Inicial   | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
| union(0,1)| | | | | | | | | |
| union(2,3)| | | | | | | | | |
| union(0,2)| | | | | | | | | |
| union(5,6)| | | | | | | | | |
| union(4,7)| | | | | | | | | |

Responde:
1. ¿Cuántos componentes hay al final?
2. ¿`connected(1, 3)`?
3. ¿`connected(4, 5)`?

In [ ]:
# Escribe aquí el id[] que obtuviste al final (luego de todas las uniones):
mi_respuesta_id = [None, None, None, None, None, None, None, None]  # ← completa los valores

# Se verifica con unittest al final de la Parte 2 (TestParte1), cuando tu QuickFind pase sus pruebas.
print(f"Tu arreglo: {mi_respuesta_id}")

---

## Parte 2 — Implementar Quick Find  💻 *(15 min)*

### Recordatorio de la clase

```
Arreglo id[]:  id[p] = identificador del componente de p

find(p)   → retorna id[p]                           O(1)
union(p,q) → cambia todos los id[i]==id[p] a id[q]  O(N)
```

Completa los `# TODO` de la clase:

In [ ]:
class QuickFind(UnionFind):
    """
    Quick Find — tu implementación.
    
    id[p] = identificador del componente al que pertenece p.
    Dos objetos p y q están conectados  <=>  id[p] == id[q]
    """

    def __init__(self, n: int):
        # TODO: inicializa el arreglo id[] de tamaño n
        #       cada objeto empieza siendo su propio componente
        self._id    = # TODO
        self._count = # TODO

    def find(self, p: int) -> int:
        # TODO: retorna el identificador del componente de p
        pass

    def union(self, p: int, q: int) -> None:
        pid = self.find(p)
        qid = self.find(q)

        if pid == qid:
            return          # ya están conectados

        # TODO: recorre self._id y cambia todos los valores pid → qid
        #       (usa un for)

        # TODO: actualiza self._count

    def count(self) -> int:
        # TODO
        pass

    def __repr__(self):
        return f"id[]={self._id}  comps={self._count}"


print("Clase definida. Ejecuta la celda de pruebas debajo.")

In [ ]:
# ── Pruebas automáticas de Quick Find (unittest) ────────────────────────────
# No modificar esta celda

class TestQuickFind(unittest.TestCase):
    """Contrato de Quick Find sobre N = 5 objetos."""

    def setUp(self):
        self.qf = QuickFind(5)          # instancia nueva para cada prueba

    def test_1_inicializacion(self):
        """__init__: cada objeto empieza siendo su propio componente"""
        self.assertEqual(self.qf._id, [0, 1, 2, 3, 4], "id[] inicial incorrecto")
        self.assertEqual(self.qf.count(), 5, "count inicial debe ser 5")

    def test_2_find_sin_uniones(self):
        """find(3) retorna 3 antes de cualquier unión"""
        self.assertEqual(self.qf.find(3), 3)

    def test_3_union_basica(self):
        """union(1,2) conecta 1 con 2 y deja count en 4"""
        self.qf.union(1, 2)
        self.assertTrue(self.qf.connected(1, 2))
        self.assertEqual(self.qf.count(), 4)

    def test_4_transitividad(self):
        """union(1,2) y union(2,3) implican connected(1,3)"""
        self.qf.union(1, 2)
        self.qf.union(2, 3)
        self.assertTrue(self.qf.connected(1, 3))

    def test_5_no_conectados(self):
        """0 y 4 no se unieron, así que no están conectados"""
        self.qf.union(1, 2)
        self.qf.union(2, 3)
        self.assertFalse(self.qf.connected(0, 4))

    def test_6_union_idempotente(self):
        """unir dos objetos ya conectados no cambia count"""
        self.qf.union(1, 2)
        self.qf.union(2, 3)
        antes = self.qf.count()
        self.qf.union(1, 3)
        self.assertEqual(self.qf.count(), antes)


resultado = unittest.main(argv=["ignorado", "TestQuickFind"], exit=False, verbosity=2)

In [ ]:
# ── Verificación de la Parte 1 — compara cada paso con tu tabla ─────────────
# La flecha ↑ marca las posiciones de id[] que cambió cada union().

qf8 = QuickFind(8)
mostrar_id(qf8._id, "Inicial  →  8 componentes")
for p, q in [(0, 1), (2, 3), (0, 2), (5, 6), (4, 7)]:
    antes = list(qf8._id)
    qf8.union(p, q)
    print()
    mostrar_id(qf8._id, f"union({p},{q})  →  {qf8.count()} componentes", anterior=antes)

print(f"\nconnected(1,3) = {qf8.connected(1, 3)}")
print(f"connected(4,5) = {qf8.connected(4, 5)}\n")


class TestParte1(unittest.TestCase):
    """Tu traza a mano de la Parte 1 contra la ejecución de QuickFind."""

    def test_tabla_a_mano(self):
        """el id[] final que escribiste coincide con el de QuickFind"""
        self.assertEqual(mi_respuesta_id, qf8._id)


resultado = unittest.main(argv=["ignorado", "TestParte1"], exit=False, verbosity=2)

---

## Parte 3 — Implementar Quick Union  💻 *(15 min)*

### Recordatorio de la clase

```
Arreglo id[]:  id[p] = PADRE de p en el árbol
               si id[p] == p  →  p es raíz

find(p)    → seguir padres hasta llegar a la raíz   O(profundidad)
union(p,q) → raíz de p apunta a raíz de q           O(profundidad)
```

**Diagrama clave:**

```
union(4, 9):    raíz(4)=4,  raíz(9)=9
                id[4] = 9

                9
                |
                4

union(3, 4):    raíz(3)=3,  raíz(4)=9
                id[3] = 9

                9
               / \
              3   4
```

In [ ]:
class QuickUnion(UnionFind):
    """
    Quick Union — tu implementación.
    
    id[p] = padre de p en el árbol.
    La raíz de cada árbol es el identificador del componente.
    """

    def __init__(self, n: int):
        # TODO: inicializa el arreglo id[] — igual que en QuickFind
        self._id    = # TODO
        self._count = # TODO

    def _root(self, p: int) -> int:
        """
        Sube por el árbol hasta encontrar la raíz.
        La raíz cumple:  id[raíz] == raíz
        """
        # TODO: implementa el ciclo while que sube hasta la raíz
        #       Pista: while self._id[p] != p:  p = ...
        pass

    def find(self, p: int) -> int:
        # TODO: retorna la raíz del árbol de p
        pass

    def union(self, p: int, q: int) -> None:
        # TODO: encuentra las raíces de p y q
        #       si son distintas, haz que la raíz de p apunte a la raíz de q
        #       actualiza self._count
        pass

    def count(self) -> int:
        # TODO
        pass

    def __repr__(self):
        return f"id[]={self._id}  comps={self._count}"


print("Clase definida. Ejecuta la celda de pruebas debajo.")

In [ ]:
# ── Pruebas automáticas de Quick Union (unittest) ───────────────────────────
# No modificar esta celda

class TestQuickUnion(unittest.TestCase):
    """Contrato de Quick Union sobre N = 6 objetos."""

    def setUp(self):
        self.qu = QuickUnion(6)         # instancia nueva para cada prueba

    def test_1_inicializacion(self):
        """__init__: cada objeto empieza siendo la raíz de su propio árbol"""
        self.assertEqual(self.qu._id, [0, 1, 2, 3, 4, 5], "id[] inicial incorrecto")
        self.assertEqual(self.qu.count(), 6, "count inicial debe ser 6")

    def test_2_root_sin_uniones(self):
        """_root(3) retorna 3 antes de cualquier unión"""
        self.assertEqual(self.qu._root(3), 3)

    def test_3_union_basica(self):
        """union(0,1) conecta 0 con 1 y deja count en 5"""
        self.qu.union(0, 1)
        self.assertTrue(self.qu.connected(0, 1))
        self.assertEqual(self.qu.count(), 5)

    def test_4_raiz_comun(self):
        """tras union(0,1) y union(1,2), 0 y 2 comparten raíz"""
        self.qu.union(0, 1)
        self.qu.union(1, 2)
        self.assertEqual(self.qu._root(0), self.qu._root(2))

    def test_5_transitividad(self):
        """union(0,1) y union(1,2) implican connected(0,2)"""
        self.qu.union(0, 1)
        self.qu.union(1, 2)
        self.assertTrue(self.qu.connected(0, 2))

    def test_6_no_conectados(self):
        """3 y 5 no se unieron, así que no están conectados"""
        self.qu.union(0, 1)
        self.qu.union(1, 2)
        self.assertFalse(self.qu.connected(3, 5))

    def test_7_union_idempotente(self):
        """unir dos objetos ya conectados no cambia count"""
        self.qu.union(0, 1)
        self.qu.union(1, 2)
        antes = self.qu.count()
        self.qu.union(0, 2)
        self.assertEqual(self.qu.count(), antes)


resultado = unittest.main(argv=["ignorado", "TestQuickUnion"], exit=False, verbosity=2)

In [ ]:
# ── Visualización: el bosque que construye tu Quick Union ───────────────────
# Cada raíz aparece entre corchetes; la sangría de un nodo es su profundidad,
# es decir, cuántos pasos da find() para llegar a la raíz.

qu_viz = QuickUnion(10)
for p, q in [(4, 3), (3, 8), (6, 5), (9, 4), (2, 1)]:
    qu_viz.union(p, q)
print(qu_viz)
print()
mostrar_bosque(qu_viz._id, "Quick Union — árbol de verificación")

---

## Parte 4 — Análisis comparativo  📊 *(10 min)*

Una vez que tus dos implementaciones pasan los tests, ejecuta el benchmark y responde las preguntas.

In [ ]:
# Benchmark: compara el tiempo de ambas implementaciones

def medir(clase, n, seed=7):
    random.seed(seed)
    uf = clase(n)
    ops = [(random.randint(0,n-1), random.randint(0,n-1)) for _ in range(n)]
    t0 = time.perf_counter()
    for p, q in ops:
        uf.union(p, q)
    for p, q in ops:
        uf.connected(p, q)
    return (time.perf_counter() - t0) * 1000  # en ms

tamanios = [200, 500, 1000, 2000, 4000, 8000]
filas = []

print(f"{'N':>6}  {'QuickFind (ms)':>16}  {'QuickUnion (ms)':>16}  {'QF / QU':>8}")
print("-" * 54)
for n in tamanios:
    tf = medir(QuickFind,  n)
    tu = medir(QuickUnion, n)
    ratio = tf/tu if tu > 0 else float('inf')
    print(f"{n:>6}  {tf:>16.2f}  {tu:>16.2f}  {ratio:>8.2f}x")
    filas += [(f"N={n} QF", tf), ("QU", tu)]

print("\nQuick Find vs Quick Union — tiempo de ejecución\n")
mostrar_barras(filas)

### ✍️ Preguntas de análisis

Responde en la celda de texto siguiente (doble clic para editar):

**Pregunta 1:** ¿Cuál algoritmo es más rápido en tu benchmark? ¿Coincide con lo esperado teóricamente?

> *Tu respuesta aquí...*

---

**Pregunta 2:** La complejidad de `union` en Quick Find es O(N) y en Quick Union también puede ser O(N) en el peor caso. ¿Por qué entonces pueden tener tiempos distintos en la práctica?

> *Tu respuesta aquí...*

---

**Pregunta 3:** Si una aplicación realiza **10 millones de `connected`** y solo **100 `union`**, ¿qué algoritmo elegirías? Justifica.

> *Tu respuesta aquí...*

---

## 🌟 Bonus — Weighted Quick Union *(si terminas antes)*

El problema de Quick Union es que el árbol puede crecer desbalanceado. La mejora es simple: **siempre colgar el árbol más pequeño debajo del más grande**.

Para esto necesitamos un arreglo auxiliar `size[]` donde `size[i]` = tamaño del árbol con raíz `i`.

```
union(p, q) con weight:
    rp = root(p),  rq = root(q)
    si size[rp] < size[rq]:
        id[rp] = rq          # el árbol chico cuelga del grande
        size[rq] += size[rp]
    si no:
        id[rq] = rp
        size[rp] += size[rq]
```

**Resultado:** la profundidad máxima es O(log N) → ¡dramáticamente mejor!

Completa la implementación:

In [ ]:
class WeightedQuickUnion(UnionFind):
    """
    Weighted Quick Union — BONUS.
    Siempre cuelga el árbol más pequeño debajo del más grande.
    Garantiza profundidad máxima O(log N).
    """

    def __init__(self, n: int):
        self._id   = list(range(n))
        self._size = [1] * n     # tamaño de cada árbol (inicialmente 1)
        self._count = n

    def _root(self, p: int) -> int:
        while self._id[p] != p:
            p = self._id[p]
        return p

    def find(self, p: int) -> int:
        return self._root(p)

    def union(self, p: int, q: int) -> None:
        rp = self._root(p)
        rq = self._root(q)
        if rp == rq:
            return

        # TODO: implementa la lógica de peso
        #       si el árbol de rp es más pequeño → cuelga rp de rq
        #       si no                            → cuelga rq de rp
        #       actualiza size[] y count
        pass

    def count(self) -> int:
        return self._count

    def max_depth(self) -> int:
        """Calcula la profundidad máxima real del bosque."""
        def depth(node):
            d = 0
            while self._id[node] != node:
                node = self._id[node]
                d += 1
            return d
        return max(depth(i) for i in range(len(self._id)))

    def __repr__(self):
        return f"id[]={self._id}  size[]={self._size}  comps={self._count}"


print("Clase definida.")

In [ ]:
# Comparación de profundidad máxima: QuickUnion vs WeightedQuickUnion
# (ejecuta solo si completaste el TODO del Bonus)

random.seed(99)
N = 500
pares = [(random.randint(0, N-1), random.randint(0, N-1)) for _ in range(N * 2)]

qu_b  = QuickUnion(N)
wqu_b = WeightedQuickUnion(N)

for p, q in pares:
    qu_b.union(p, q)
    wqu_b.union(p, q)

print(f"QuickUnion         — profundidad máxima: {qu_b.max_depth()}")
print(f"WeightedQuickUnion — profundidad máxima: {wqu_b.max_depth()}")
import math
print(f"log₂({N}) = {math.log2(N):.1f}  (cota teórica de Weighted)")

---

## ✅ Checklist de entrega

Antes de subir tu notebook, verifica:

- [ ] Tabla de la Parte 1 completada
- [ ] `TestQuickFind` termina en `OK` (6 pruebas) y `TestParte1` también
- [ ] `TestQuickUnion` termina en `OK` (7 pruebas)
- [ ] Las 3 preguntas de análisis respondidas
- [ ] *(Opcional)* `WeightedQuickUnion` implementado
- [ ] Notebook ejecutado de arriba a abajo sin errores (`Kernel → Restart & Run All`)

---

**Nombre:** ___________________________  
**Fecha:** ___________________________